# Simple RNN text classification

Train a small RNN to label short movie comments as **positive** or **negative**. This is a learning example with a deliberately tiny dataset.

In [3]:
import pandas as pd
df = pd.read_csv('IMDB Dataset.csv')
df = df.sample(n=2500, random_state=42)
df = df.reset_index(drop=True)
print(df.head())

                                              review sentiment
0  I really liked this Summerslam due to the look...  positive
1  Not many television shows appeal to quite as m...  positive
2  The film quickly gets to a major chase scene w...  negative
3  Jane Austen would definitely approve of this o...  positive
4  Expectations were somewhat high for me when I ...  negative


In [4]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(df[['review', 'sentiment']].head())

                                              review  sentiment
0  I really liked this Summerslam due to the look...          1
1  Not many television shows appeal to quite as m...          1
2  The film quickly gets to a major chase scene w...          0
3  Jane Austen would definitely approve of this o...          1
4  Expectations were somewhat high for me when I ...          0


In [5]:
# Import the standard splitting function from the scikit-learn library
from sklearn.model_selection import train_test_split

# We pass in our text data (X) and our labels (y)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['review'].values,    # The input features (the movie reviews)
    df['sentiment'].values, # The target labels (the 1s and 0s)
    test_size=0.2,          # Reserve 20% of the data for the test set
    random_state=42         # Set a seed so the random split is identical every time
)

# (Optional) Verify the sizes of the resulting datasets
print(f"Training reviews: {len(X_train_text)}")
print(f"Testing reviews: {len(X_test_text)}")

Training reviews: 2000
Testing reviews: 500


In [6]:
# Import the tiktoken library (Make sure to run: pip install tiktoken in your terminal first)
import tiktoken

# Load the exact encoding requested in the image instructions
enc = tiktoken.get_encoding("cl100k_base")

# Convert every text review in our training set into a list of integer IDs
X_train_tokens = [enc.encode(review) for review in X_train_text]

# Do the exact same thing for the testing set
X_test_tokens = [enc.encode(review) for review in X_test_text]

# (Optional) Print the first review to see what the token IDs look like
print("Original text preview:", X_train_text[0][:50], "...")
print("Token IDs:", X_train_tokens[0][:10], "...")

Original text preview: It's a little disconcerting to have a character na ...
Token IDs: [2181, 596, 264, 2697, 834, 89405, 287, 311, 617, 264] ...


In [7]:
import numpy as np

# Set the strict sequence length we want for every review
MAX_SEQ_LENGTH = 150

def pad_or_truncate(token_lists, max_length):
    fixed_sequences = []

    for tokens in token_lists:
        if len(tokens) > max_length:
            # TRUNCATE: If the review is longer than 150 tokens, chop off the end
            fixed_tokens = tokens[:max_length]
        else:
            # PAD: If the review is shorter than 150, add zeros to the end to fill the space
            fixed_tokens = tokens + [0] * (max_length - len(tokens))

        fixed_sequences.append(fixed_tokens)

    # Convert the final list of lists into a 2D NumPy array
    return np.array(fixed_sequences)

# Apply this function to the tokenized lists we created in Step 4
X_train_padded = pad_or_truncate(X_train_tokens, MAX_SEQ_LENGTH)
X_test_padded = pad_or_truncate(X_test_tokens, MAX_SEQ_LENGTH)

# (Optional) Verify the shape. It should output (2000, 150) meaning 2000 reviews, all exactly 150 tokens long.
print(f"Training data shape: {X_train_padded.shape}")

Training data shape: (2000, 150)


In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Convert the input features (padded tokens) into Tensors
# We use dtype=torch.long (64-bit integer) because these are ID numbers
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

# 2. Convert the target labels (0s and 1s) into Tensors
# We use dtype=torch.float32 (decimals) because the loss function calculates decimal probabilities
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# 3. Combine the inputs and labels into a Dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# 4. Create DataLoaders to feed the data in small batches
# batch_size=32 means the model will look at 32 reviews at a time before updating its weights
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Tensors and DataLoaders created successfully!")

Tensors and DataLoaders created successfully!


In [9]:
import torch.nn as nn

class IMDBClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        # 1. The Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # 2. The RNN Layer (batch_first=True tells it that our data is shaped [batch_size, sequence_length])
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

        # 3. The Linear Output Layer
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Step A: Convert token IDs into dense vectors
        embedded = self.embedding(x)

        # Step B: Pass the sequence of vectors through the RNN
        # 'out' contains the memory at every single step, 'hidden' is the final memory state
        out, hidden = self.rnn(embedded)

        # Step C: Take the very last memory state (hidden[-1]) and pass it to the linear layer
        # .squeeze(1) flattens the output from [32, 1] to just [32] to match our labels
        final_prediction = self.linear(hidden[-1]).squeeze(1)

        return final_prediction

# Instantiate the model
# The cl100k_base tokenizer we used in Step 4 has exactly 100,277 unique tokens in its vocabulary
VOCAB_SIZE = 100277
EMBED_DIM = 64  # Size of the vector representing each word
HIDDEN_DIM = 32 # Size of the RNN's internal memory
OUTPUT_DIM = 1  # We want a single number out (closer to 1 = positive, closer to 0 = negative)

model = IMDBClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)
print(model)

IMDBClassifier(
  (embedding): Embedding(100277, 64)
  (rnn): RNN(64, 32, batch_first=True)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


In [11]:
import torch.nn as nn

class IMDBClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        # 1. The Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # 2. The RNN Layer (batch_first=True tells it that our data is shaped [batch_size, sequence_length])
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

        # 3. The Linear Output Layer
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Step A: Convert token IDs into dense vectors
        embedded = self.embedding(x)

        # Step B: Pass the sequence of vectors through the RNN
        # 'out' contains the memory at every single step, 'hidden' is the final memory state
        out, hidden = self.rnn(embedded)

        # Step C: Take the very last memory state (hidden[-1]) and pass it to the linear layer
        # .squeeze(1) flattens the output from [32, 1] to just [32] to match our labels
        final_prediction = self.linear(hidden[-1]).squeeze(1)

        return final_prediction

# Instantiate the model
# The cl100k_base tokenizer we used in Step 4 has exactly 100,277 unique tokens in its vocabulary
VOCAB_SIZE = 100277
EMBED_DIM = 64  # Size of the vector representing each word
HIDDEN_DIM = 32 # Size of the RNN's internal memory
OUTPUT_DIM = 1  # We want a single number out (closer to 1 = positive, closer to 0 = negative)

model = IMDBClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)
print(model)

IMDBClassifier(
  (embedding): Embedding(100277, 64)
  (rnn): RNN(64, 32, batch_first=True)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


In [12]:
import torch.optim as optim

# 1. Choose the Loss Function and Optimizer
# BCEWithLogitsLoss is the mathematically correct choice for binary classification (0 or 1) in PyTorch
criterion = nn.BCEWithLogitsLoss()

# Adam is a highly efficient algorithm that adjusts the model's weights
# lr=0.001 is the "learning rate" (how big of a step to take when adjusting weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Set the number of times to loop through the entire training dataset
epochs = 5

print("Starting Training...")
for epoch in range(epochs):
    # Set the model to training mode (enables specific layers like dropout if you had them)
    model.train()
    total_loss = 0

    # Loop through the DataLoader we created in Step 6 (batches of 32)
    for batch_x, batch_y in train_loader:

        # Step A: Clear old gradients so they don't stack up from previous batches
        optimizer.zero_grad()

        # Step B: Forward Pass (ask the model to guess if the reviews are positive or negative)
        predictions = model(batch_x)

        # Step C: Calculate the Loss (how far off the guesses were from the actual labels)
        loss = criterion(predictions, batch_y)

        # Step D: Backward Pass (calculate which weights need to change to reduce the error)
        loss.backward()

        # Step E: Optimize (actually update the model's weights)
        optimizer.step()

        # Add the current batch's loss to our running total
        total_loss += loss.item()

    # Print the average loss periodically (at the end of each epoch)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {avg_loss:.4f}")

Starting Training...
Epoch 1/5 - Average Loss: 0.6996
Epoch 2/5 - Average Loss: 0.6855
Epoch 3/5 - Average Loss: 0.6738
Epoch 4/5 - Average Loss: 0.6564
Epoch 5/5 - Average Loss: 0.6296


In [13]:
import torch

# 1. Set the model to evaluation mode
# This tells the model we are no longer training. It disables certain layers
# (like Dropout, if you add them later) that are only meant for the training phase.
model.eval()

correct_predictions = 0
total_samples = 0

print("Evaluating model on test data...")

# 2. Turn off gradient calculation
# Since we are not updating weights, we use torch.no_grad().
# This saves a massive amount of memory and makes the code run much faster.
with torch.no_grad():
    # Loop through the test data in batches
    for batch_x, batch_y in test_loader:

        # Step A: Get the raw output numbers (logits) from the model
        raw_scores = model(batch_x)

        # Step B: Convert raw scores into probabilities using the Sigmoid function
        # This squashes the numbers into a range strictly between 0.0 and 1.0
        probabilities = torch.sigmoid(raw_scores)

        # Step C: Convert probabilities to binary guesses
        # If probability is 0.5 or higher, guess 1 (positive). Otherwise, guess 0 (negative).
        predictions = (probabilities >= 0.5).float()

        # Step D: Compare the model's guesses to the actual answers
        # Count how many were exactly right and add them to our running total
        correct_predictions += (predictions == batch_y).sum().item()

        # Keep track of the total number of reviews we have tested so far
        total_samples += batch_y.size(0)

# 3. Calculate and print the final accuracy percentage
accuracy = (correct_predictions / total_samples) * 100
print(f"Final Test Accuracy: {accuracy:.2f}%")

Evaluating model on test data...
Final Test Accuracy: 47.40%


In [14]:
import torch

def predict_sentiment(review_text):
    # 1. Set the model to evaluation mode
    model.eval()

    # 2. Tokenize the custom text using our tiktoken encoder
    tokens = enc.encode(review_text)

    # 3. Pad or truncate to match our strict 150-token requirement
    if len(tokens) > MAX_SEQ_LENGTH:
        tokens = tokens[:MAX_SEQ_LENGTH]
    else:
        tokens = tokens + [0] * (MAX_SEQ_LENGTH - len(tokens))

    # 4. Convert to a PyTorch Tensor
    # We use .unsqueeze(0) to add a fake "batch" dimension.
    # The model expects data shaped [batch_size, sequence_length], so this makes it [1, 150]
    input_tensor = torch.tensor(tokens, dtype=torch.long).unsqueeze(0)

    # 5. Make the prediction
    with torch.no_grad():
        raw_score = model(input_tensor)
        # Convert the raw logit into a probability between 0 and 1
        probability = torch.sigmoid(raw_score).item()

    # 6. Print the final result
    print(f"Review: '{review_text}'")
    if probability >= 0.5:
        print(f"Prediction: POSITIVE (Confidence: {probability:.2f})\n")
    else:
        # For negative reviews, we invert the probability to show how confident it is that it's negative
        print(f"Prediction: NEGATIVE (Confidence: {1 - probability:.2f})\n")

# --- Let's test it out! ---
predict_sentiment("This movie was absolutely fantastic! The acting was brilliant and the story kept me hooked.")
predict_sentiment("What a total waste of time. The plot made zero sense and the characters were incredibly boring.")

Review: 'This movie was absolutely fantastic! The acting was brilliant and the story kept me hooked.'
Prediction: NEGATIVE (Confidence: 0.50)

Review: 'What a total waste of time. The plot made zero sense and the characters were incredibly boring.'
Prediction: NEGATIVE (Confidence: 0.50)



In [15]:
import torch

# -----------------------------------------
# 1. Saving the Model
# -----------------------------------------
# We save the 'state_dict', which is a Python dictionary containing all the
# learned weights and biases of our specific model layers.
# The standard file extension for PyTorch models is .pth or .pt
torch.save(model.state_dict(), 'imdb_rnn_model.pth')
print("Model weights saved to 'imdb_rnn_model.pth'")

# -----------------------------------------
# 2. Loading the Model (For Future Use)
# -----------------------------------------
# To load a model, you first have to create a "blank" version of the architecture
loaded_model = IMDBClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)

# Next, you inject the saved weights into this blank model
loaded_model.load_state_dict(torch.load('imdb_rnn_model.pth'))

# Finally, always set it to evaluation mode before testing!
loaded_model.eval()
print("Model weights loaded successfully!")

# You can now use `loaded_model` exactly like we used `model` in Step 10


Model weights saved to 'imdb_rnn_model.pth'
Model weights loaded successfully!


In [16]:
import torch.nn as nn

class IMDB_LSTM_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # CHANGED: We swap nn.RNN for nn.LSTM
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)

        # CHANGED: LSTMs output a more complex memory state.
        # It returns 'out', plus a tuple containing BOTH a 'hidden' state and a 'cell' state.
        out, (hidden, cell) = self.lstm(embedded)

        # We still just take the final hidden state and pass it to the linear layer
        final_prediction = self.linear(hidden[-1]).squeeze(1)

        return final_prediction

# Instantiate the new, smarter model
lstm_model = IMDB_LSTM_Classifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)
print(lstm_model)

IMDB_LSTM_Classifier(
  (embedding): Embedding(100277, 64)
  (lstm): LSTM(64, 32, batch_first=True)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


In [17]:
import torch.nn as nn

class IMDB_LSTM_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # We can add dropout directly inside the LSTM for its internal connections...
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, dropout=dropout_rate if hidden_dim > 1 else 0)

        # ...and we add a standalone Dropout layer before the final output
        self.dropout = nn.Dropout(dropout_rate)

        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)

        out, (hidden, cell) = self.lstm(embedded)

        # CHANGED: We pass the final hidden state through the dropout layer
        # before giving it to the linear layer
        dropped_out = self.dropout(hidden[-1])

        final_prediction = self.linear(dropped_out).squeeze(1)

        return final_prediction

# Instantiate the model with a 50% dropout rate
lstm_model_with_dropout = IMDB_LSTM_Classifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, dropout_rate=0.5)
print(lstm_model_with_dropout)

IMDB_LSTM_Classifier(
  (embedding): Embedding(100277, 64)
  (lstm): LSTM(64, 32, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [18]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import tiktoken

# ==========================================
# 1. DATA PREPARATION
# ==========================================
print("Loading and preparing data...")
df = pd.read_csv('IMDB Dataset.csv')
df = df.sample(n=2500, random_state=42).reset_index(drop=True)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['review'].values,
    df['sentiment'].values,
    test_size=0.2,
    random_state=42
)

# ==========================================
# 2. TOKENIZATION AND PADDING
# ==========================================
print("Tokenizing text with tiktoken...")
enc = tiktoken.get_encoding("cl100k_base")
MAX_SEQ_LENGTH = 150

def tokenize_and_pad(texts, max_length):
    padded_sequences = []
    for text in texts:
        tokens = enc.encode(text)
        if len(tokens) > max_length:
            tokens = tokens[:max_length]
        else:
            tokens = tokens + [0] * (max_length - len(tokens))
        padded_sequences.append(tokens)
    return np.array(padded_sequences)

X_train_padded = tokenize_and_pad(X_train_text, MAX_SEQ_LENGTH)
X_test_padded = tokenize_and_pad(X_test_text, MAX_SEQ_LENGTH)

# ==========================================
# 3. TENSORS AND DATALOADERS
# ==========================================
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)

# ==========================================
# 4. MODEL ARCHITECTURE (LSTM + Dropout)
# ==========================================
class IMDB_LSTM_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, dropout=dropout_rate if hidden_dim > 1 else 0)
        self.dropout = nn.Dropout(dropout_rate)
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        out, (hidden, cell) = self.lstm(embedded)
        dropped_out = self.dropout(hidden[-1])
        return self.linear(dropped_out).squeeze(1)

# Initialize model
VOCAB_SIZE = enc.n_vocab
model = IMDB_LSTM_Classifier(vocab_size=VOCAB_SIZE, embed_dim=64, hidden_dim=32, output_dim=1)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5

print("\nStarting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {total_loss / len(train_loader):.4f}")

# ==========================================
# 6. EVALUATION
# ==========================================
print("\nEvaluating model on test data...")
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        raw_scores = model(batch_x)
        probabilities = torch.sigmoid(raw_scores)
        predictions = (probabilities >= 0.5).float()
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)

print(f"Final Test Accuracy: {(correct / total) * 100:.2f}%")

Loading and preparing data...
Tokenizing text with tiktoken...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)



Starting Training...
Epoch 1/5 - Average Loss: 0.6968
Epoch 2/5 - Average Loss: 0.6879
Epoch 3/5 - Average Loss: 0.6824
Epoch 4/5 - Average Loss: 0.6751
Epoch 5/5 - Average Loss: 0.6658

Evaluating model on test data...
Final Test Accuracy: 51.00%
